# 05. 횡단보도 API 전처리

공공데이터포털 `전국횡단보도표준데이터` API 전체를 수집하고,
229개 행정구역 기준으로 지역명을 통일한 뒤 동일 설치지점 중복을 제거합니다.

**출력**
- `../data_processed/crosswalk_count_final.csv`

> API에 데이터가 없는 지역은 횡단보도 0개로 해석하지 않습니다.

In [1]:
import math
import time
import requests
import pandas as pd
from urllib.parse import unquote

URL = "https://api.data.go.kr/openapi/tn_pubr_public_crosswalk_api"

ENCODING_KEY = "dJ86KOJSsPyb6606Leb2zS7oit1l%2BHE1Vo2nmwAzyHvB7I1UdP%2B0guI2VeyfG9zFcQS7pvg2Ml11qoh8LcRIoA%3D%3D"
service_key = unquote(ENCODING_KEY)


## 1) API 전체 수집

In [2]:
params = {
    "serviceKey": service_key,
    "pageNo": 1,
    "numOfRows": 100,
    "type": "json"
}

res = requests.get(URL, params=params, timeout=30)
res.raise_for_status()

data = res.json()
total_count = int(data["body"]["totalCount"])

print("전체 건수:", total_count)

전체 건수: 58839


In [3]:
num_rows = 1000
total_pages = math.ceil(total_count / num_rows)
all_records = []

for page in range(1, total_pages + 1):
    params = {
        "serviceKey": service_key,
        "pageNo": page,
        "numOfRows": num_rows,
        "type": "json"
    }

    response = requests.get(URL, params=params, timeout=30)
    response.raise_for_status()

    records = (
        response.json()["body"]
        .get("items", {})
        .get("item", [])
    )

    if isinstance(records, dict):
        records = [records]

    all_records.extend(records)

    if page % 10 == 0 or page == total_pages:
        print(f"{page}/{total_pages} 완료 - 누적 {len(all_records)}건")

    time.sleep(0.05)

crosswalk = pd.DataFrame(all_records)
print("수집 완료:", crosswalk.shape)
display(crosswalk.head())

10/59 완료 - 누적 10000건
20/59 완료 - 누적 20000건
30/59 완료 - 누적 30000건
40/59 완료 - 누적 40000건
50/59 완료 - 누적 50000건
59/59 완료 - 누적 58839건
수집 완료: (58839, 28)


,ctprvnNm,signguNm,roadNm,rdnmadr,lnmadr,crslkManageNo,crslkKnd,bcyclCrslkCmbnatYn,highlandYn,latitude,...,redSgngnrTime,tfcilndYn,ftpthLowerYn,brllBlckYn,cnctrLghtFcltyYn,institutionNm,phoneNumber,referenceDate,insttCode,insttNm
0,충청남도,천안시,청수16로,,충청남도 천안시 동남구 청수동 431,900106,01,,N,36.789203,...,,N,Y,N,,충청남도 천안시청,041-521-5885,2025-11-28,4490000,충청남도 천안시
1,충청남도,천안시,청수16로,,충청남도 천안시 동남구 청수동 431,900107,01,,Y,36.788971,...,,N,Y,Y,,충청남도 천안시청,041-521-5885,2025-11-28,4490000,충청남도 천안시
2,충청남도,천안시,청수6로,,충청남도 천안시 동남구 청당동 958,900108,01,,Y,36.788798,...,,N,Y,Y,,충청남도 천안시청,041-521-5885,2025-11-28,4490000,충청남도 천안시
3,충청남도,천안시,청수6로,,충청남도 천안시 동남구 청수동 431,900109,01,,N,36.789457,...,,N,Y,Y,,충청남도 천안시청,041-521-5885,2025-11-28,4490000,충청남도 천안시
4,충청남도,천안시,청수6로,,충청남도 천안시 동남구 청당동 956,900110,01,,N,36.784948,...,,N,Y,Y,,충청남도 천안시청,041-521-5885,2025-11-28,4490000,충청남도 천안시


## 2) 지역명 229개 기준으로 정규화

In [4]:
PARENT_CITIES = [
    "수원시", "성남시", "안양시", "부천시",
    "안산시", "고양시", "용인시",
    "천안시", "청주시", "전주시",
    "포항시", "창원시"
]
GWANGJU_GU = {"동구", "서구", "남구", "북구", "광산구"}

def normalize_api_region(df):
    out = df.copy()

    out["ctprvnNm"] = out["ctprvnNm"].astype(str).str.strip()
    out["signguNm"] = out["signguNm"].astype(str).str.strip()

    out["SIDO_NM"] = out["ctprvnNm"]
    out["SIGUNGU_NM"] = out["signguNm"]

    # 데이터에 나타나는 통합 행정명칭 보정
    mask = out["SIDO_NM"] == "전남광주통합특별시"
    out.loc[
        mask & out["SIGUNGU_NM"].isin(GWANGJU_GU),
        "SIDO_NM"
    ] = "광주광역시"
    out.loc[
        mask & ~out["SIGUNGU_NM"].isin(GWANGJU_GU),
        "SIDO_NM"
    ] = "전라남도"

    # 일반시 아래 구 → 229개 기준 상위 시
    compact = out["SIGUNGU_NM"].str.replace(" ", "", regex=False)
    for city in PARENT_CITIES:
        city_mask = compact.str.startswith(city, na=False)
        out.loc[city_mask, "SIGUNGU_NM"] = city

    # 세종
    sejong_mask = (
        (out["SIDO_NM"] == "세종특별자치시") &
        (out["SIGUNGU_NM"].isin(["없음", "세종특별자치시"]))
    )
    out.loc[sejong_mask, "SIGUNGU_NM"] = "세종시"

    return out

In [5]:
cw = normalize_api_region(crosswalk)

mapping = pd.read_csv(
    "../data_raw/sgis_sigungu_252_to_229_mapping.csv"
)

region_229 = (
    mapping[["SIDO_NM", "sgg_nm_229"]]
    .drop_duplicates()
    .rename(columns={"sgg_nm_229": "SIGUNGU_NM"})
)

coverage = (
    region_229
    .merge(
        cw[["SIDO_NM", "SIGUNGU_NM"]].drop_duplicates(),
        on=["SIDO_NM", "SIGUNGU_NM"],
        how="left",
        indicator=True
    )
)

print("전국 기준 지역:", len(region_229))
print("횡단보도 API 데이터 있는 지역:",
      (coverage["_merge"] == "both").sum())
print("횡단보도 API 데이터 없는 지역:",
      (coverage["_merge"] == "left_only").sum())

전국 기준 지역: 229
횡단보도 API 데이터 있는 지역: 59
횡단보도 API 데이터 없는 지역: 170


## 3) 동일 설치지점 중복 제거 및 지역별 집계

In [6]:
cw_unique = cw.drop_duplicates(
    subset=[
        "SIDO_NM",
        "SIGUNGU_NM",
        "crslkManageNo",
        "latitude",
        "longitude"
    ]
).copy()

crosswalk_count_final = (
    cw_unique
    .groupby(["SIDO_NM", "SIGUNGU_NM"])
    .size()
    .reset_index(name="crosswalk_count")
)

print("원본 API 행:", len(cw))
print("중복 정리 후 설치지점:", len(cw_unique))
print("데이터 제공 지역:", len(crosswalk_count_final))
display(crosswalk_count_final.head())

원본 API 행: 58839
중복 정리 후 설치지점: 57004
데이터 제공 지역: 59


,SIDO_NM,SIGUNGU_NM,crosswalk_count
0,강원특별자치도,동해시,1105
1,강원특별자치도,정선군,43
2,강원특별자치도,태백시,4
3,강원특별자치도,평창군,92
4,경기도,과천시,124


In [7]:
crosswalk_count_final.to_csv(
    "../data_processed/crosswalk_count_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

저장 완료
